# Chuỗi ngày học
Dự đoán số chuỗi của các ngày 5, 6, 7, 9 và 10/01. Chạy từng ô bằng Shift+Enter.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'run_streak.py').is_file())
sys.path.insert(0, str(root))
from streak_lab.data import connect
from streak_lab.analysis import checked, read_sql, export
with connect() as connection:
    display(pd.read_sql_query('SELECT * FROM study_sessions WHERE learner_id=1 ORDER BY session_id', connection))

Gom các buổi về một dòng mỗi ngày. Trừ số thứ tự khỏi ngày để các ngày liền nhau có cùng khóa nhóm.

In [ ]:
with connect() as connection:
    display(pd.read_sql_query("""WITH days AS (
        SELECT DISTINCT studied_on FROM study_sessions WHERE learner_id=1
    ), numbered AS (
        SELECT *, ROW_NUMBER() OVER (ORDER BY studied_on) AS n FROM days
    )
    SELECT *, date(studied_on, printf('-%d days',n)) AS group_day FROM numbered""", connection))
print(read_sql('streaks'))
with connect() as connection:
    tables = checked(connection)
sample = tables['summary'].iloc[0]
assert (sample.sessions, sample.study_days) == (7, 5)
assert tables['streaks'].query('learner_id == 1').days.tolist() == [3, 2]
display(tables['streaks'].query('learner_id == 1'))

Tuần bắt đầu thứ Hai. Người 02 học Chủ nhật và thứ Hai liền nhau. So sánh chuỗi với hai tuần đồng hạng.

In [ ]:
display(tables['weeks'].query('learner_id <= 2'))
display(tables['summary'].head())
export()
from IPython.display import Image
display(Image(filename=str(root/'outputs/study_weeks.png')))